# 02 — Data Cleaning & Preprocessing

## Objective

This notebook converts the raw Bengaluru Urban groundwater dataset into a
validated and reproducible dataset suitable for EDA, feature engineering and
machine learning.

The previous notebook identified duplicate records, missing values, irregular
time gaps and varying station coverage. Here, these issues will be investigated
and handled systematically.

### Preprocessing Principles

- Preserve the original raw dataset.
- Remove only confirmed duplicate records.
- Do not blindly fabricate missing groundwater observations.
- Investigate suspicious values before removing them.
- Preserve station and geographic information.
- Record the reasoning behind every preprocessing decision.
- Save the final cleaned dataset for use in later notebooks.

### Pipeline

Raw Data
→ Duplicate Handling
→ Missing-Value Analysis
→ Value Validation
→ Time-Gap Analysis
→ Geographic Validation
→ Final Cleaning
→ Clean Dataset

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd().resolve()
raw_path = None
for candidate in [project_root, project_root.parent]:
    path = candidate / "data" / "raw" / "Bengaluru_dataset.csv"
    if path.exists():
        raw_path = path
        break

if raw_path is None:
    raise FileNotFoundError("Bengaluru_dataset.csv not found under data/raw/.")

# Load raw dataset
df = pd.read_csv(raw_path)

print("Raw dataset shape:", df.shape)
print("Columns:", df.shape[1])
print("Source:", raw_path)

Raw dataset shape: (100881, 22)
Columns: 22
Source: /workspaces/groundwater-ai/data/raw/Bengaluru_dataset.csv


In [2]:
print("Duplicate rows:", df.duplicated().sum())

print(
    "Duplicate Station + Timestamp combinations:",
    df.duplicated(
        subset=["Station", "Data Acquisition Time"]
    ).sum()
)

Duplicate rows: 1434
Duplicate Station + Timestamp combinations: 1434


## Duplicate Record Handling

Duplicate analysis showed 1,434 duplicate rows and 1,434 duplicate Station–Timestamp combinations.  
Earlier analysis confirmed that these duplicate records contain identical groundwater-level values.

Therefore, the duplicate records can be safely removed without losing unique groundwater observations.

Next, the cleaned dataset will be checked for missing values, invalid readings, outliers, and remaining temporal inconsistencies.

In [3]:
# Remove exact duplicate rows
df_clean = df.drop_duplicates().copy()

# Remove duplicate Station + Timestamp combinations
df_clean = df_clean.drop_duplicates(
    subset=["Station", "Data Acquisition Time"],
    keep="first"
).copy()

print("Original rows :", len(df))
print("Cleaned rows  :", len(df_clean))
print("Rows removed  :", len(df) - len(df_clean))

print("\nRemaining duplicate Station + Timestamp combinations:",
      df_clean.duplicated(
          subset=["Station", "Data Acquisition Time"]
      ).sum())

Original rows : 100881
Cleaned rows  : 99447
Rows removed  : 1434

Remaining duplicate Station + Timestamp combinations: 0


### Findings

- 1,434 duplicate Station–Timestamp records were identified.
- Previous analysis confirmed that all duplicate groups had identical groundwater-level values.
- Duplicate records were removed while retaining one valid observation per Station–Timestamp combination.
- Final verification showed **0 remaining duplicate Station–Timestamp combinations**.
- No unique groundwater observation was lost during this step.

## Missing Value Analysis

After removing duplicate observations, the cleaned dataset is examined for missing values.

The objective is to identify:
- Which columns contain missing values
- How many values are missing
- The percentage of missing values
- Whether the missing values affect features required for groundwater-level prediction

Missing values will not be filled blindly; the treatment will depend on the role and extent of missingness in each column.

In [4]:
# Missing value analysis after duplicate removal

missing_summary = pd.DataFrame({
    "missing_count": df_clean.isnull().sum(),
    "missing_percent": (df_clean.isnull().sum() / len(df_clean)) * 100
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_count", ascending=False)

print("Columns containing missing values:")
display(missing_summary)

Columns containing missing values:


,missing_count,missing_percent
RL_MSL,29537,29.701248


In [5]:
# Check missing values in the main project-related columns

important_columns = [
    "Station",
    "Latitude",
    "Longitude",
    "Data Acquisition Time",
    "Groundwater Level Telemetry 6 Hourly (meter)",
    "RL_MSL"
]

print("Missing values in important columns:\n")
print(df_clean[important_columns].isnull().sum())

Missing values in important columns:

Station                                             0
Latitude                                            0
Longitude                                           0
Data Acquisition Time                               0
Groundwater Level Telemetry 6 Hourly (meter)        0
RL_MSL                                          29537
dtype: int64


## RL_MSL Missingness Investigation

The only missing values occur in `RL_MSL`, with 29.70% of observations affected.

Since groundwater level, station identity, coordinates, and timestamps are complete, the missing `RL_MSL` values do not directly affect the prediction target.

Before deciding how to treat `RL_MSL`, its missingness is analyzed across stations to determine whether the missing values are concentrated at specific monitoring locations.

In [6]:
# Analyze RL_MSL missingness by station

rl_msl_station = (
    df_clean.groupby("Station")
    .agg(
        total_records=("RL_MSL", "size"),
        missing_RL_MSL=("RL_MSL", lambda x: x.isna().sum())
    )
)

rl_msl_station["missing_percent"] = (
    rl_msl_station["missing_RL_MSL"]
    / rl_msl_station["total_records"] * 100
)

rl_msl_station = rl_msl_station.sort_values(
    "missing_percent", ascending=False
)

display(rl_msl_station)

,total_records,missing_RL_MSL,missing_percent
Station,,,
Doddakannahalli,4345,4345,100.0
Kethohalli,4656,4656,100.0
Yelahanka_1,4623,4623,100.0
Rajanukunte,4107,4107,100.0
Mahadevapura_2,3485,3485,100.0
Jakkur_1,3721,3721,100.0
Laggere,4600,4600,100.0
Anekal_1,3007,0,0.0
Adakamaranahalli,4596,0,0.0


## 3. Station-wise Analysis of Missing RL_MSL

`RL_MSL` contains 29,537 missing values (29.70% of the dataset).

The missingness is completely station-specific:
- 7 stations have 100% missing `RL_MSL`.
- The remaining 18 stations have no missing `RL_MSL`.

This indicates that `RL_MSL` is unavailable for specific stations rather than randomly missing.

**Decision:** Do not blindly impute `RL_MSL`. Since it is station-level reference/elevation metadata and not the groundwater-level target, available values will be retained, while completely missing stations will be handled separately if `RL_MSL` is required for later analysis or modelling.

In [7]:
# Check RL_MSL consistency within each station

rl_msl_consistency = (
    df.groupby('Station')['RL_MSL']
      .agg(
          unique_values='nunique',
          non_missing='count',
          min_value='min',
          max_value='max'
      )
      .sort_values('unique_values', ascending=False)
)

rl_msl_consistency

,unique_values,non_missing,min_value,max_value
Station,,,,
Adakamaranahalli,1,4596,849.0,849.0
Anekal_1,1,3007,927.0,927.0
Attibele_1,1,3594,883.0,883.0
Avalahalli,1,3110,877.0,877.0
Bagalagunte,1,4684,888.0,888.0
Beguru_1,1,3556,891.0,891.0
Byadarahalli,1,4612,876.0,876.0
Chandapura_1,1,4622,880.0,880.0
Devarabeesanahalli_1,1,3414,874.0,874.0


## 4. RL_MSL Consistency Check

`RL_MSL` is constant within each station:
- All 18 stations with available values have exactly one unique `RL_MSL`.
- The remaining 7 stations have no `RL_MSL` data.

This confirms that `RL_MSL` is station-level metadata rather than a time-varying feature.

**Decision:** Preserve the available station-level `RL_MSL` values. Do not perform row-wise interpolation or time-series imputation. The 7 stations with completely missing `RL_MSL` will remain missing unless this feature is specifically required for modelling.

In [8]:
# Check missing values in the groundwater-level target

target_col = 'Groundwater Level Telemetry 6 Hourly (meter)'

target_missing = df[target_col].isna().sum()
target_total = len(df)
target_missing_percent = (target_missing / target_total) * 100

print(f"Target column: {target_col}")
print(f"Total records: {target_total}")
print(f"Missing target values: {target_missing}")
print(f"Missing percentage: {target_missing_percent:.2f}%")

Target column: Groundwater Level Telemetry 6 Hourly (meter)
Total records: 100881
Missing target values: 0
Missing percentage: 0.00%


## 5. Target Variable Missingness Check

The groundwater-level target contains **0 missing values** across all 100,881 records.

Therefore:
- No target imputation is required.
- No records need to be removed due to missing target values.
- The complete target series can be retained for subsequent preprocessing and modelling.

**Decision:** Keep all target observations unchanged.

In [9]:
# Basic statistical analysis of the groundwater-level target

target_col = 'Groundwater Level Telemetry 6 Hourly (meter)'

target_stats = df[target_col].describe()

print("Groundwater Level Target Statistics:")
print(target_stats)

Groundwater Level Target Statistics:
count    1.008810e+05
mean     5.684454e+01
std      2.391772e+04
min     -1.901060e+02
25%     -2.853200e+01
50%     -1.828500e+01
75%     -1.042000e+01
max      7.563528e+06
Name: Groundwater Level Telemetry 6 Hourly (meter), dtype: float64


## 6. Target Value Quality & Outlier Investigation

The groundwater-level target contains no missing values, so the next step is to assess
whether the recorded values are statistically consistent and identify potential
outliers or sensor/data-entry anomalies.

The unusually large maximum value and high standard deviation indicate the presence
of extreme observations. These values will be investigated before deciding whether
they should be removed, corrected, or retained.

**Decision:** Do not remove outliers blindly. First identify their frequency,
station distribution, and magnitude.

In [10]:
target_col = "Groundwater Level Telemetry 6 Hourly (meter)"

# Basic outlier thresholds using IQR
Q1 = df[target_col].quantile(0.25)
Q3 = df[target_col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (
    (df[target_col] < lower_bound) |
    (df[target_col] > upper_bound)
)

outliers = df[outlier_mask].copy()

print("IQR Outlier Analysis")
print("-" * 40)
print(f"Q1: {Q1:.3f}")
print(f"Q3: {Q3:.3f}")
print(f"IQR: {IQR:.3f}")
print(f"Lower bound: {lower_bound:.3f}")
print(f"Upper bound: {upper_bound:.3f}")
print(f"Total outliers: {len(outliers)}")
print(f"Outlier percentage: {len(outliers) / len(df) * 100:.2f}%")

IQR Outlier Analysis
----------------------------------------
Q1: -28.532
Q3: -10.420
IQR: 18.112
Lower bound: -55.700
Upper bound: 16.748
Total outliers: 10616
Outlier percentage: 10.52%


## 7. Outlier Distribution & Station-wise Investigation

The IQR analysis identified 10,616 potential outliers (10.52% of all
groundwater-level observations). Before removing or transforming these values,
their distribution across monitoring stations is investigated.

This step checks whether the outliers are concentrated in specific stations
and whether they represent isolated extreme measurements or a broader pattern.

**Decision:** Outliers will not be removed at this stage. Their station-wise
distribution will be examined first to determine an appropriate treatment.

In [11]:
# Identify IQR outliers
outlier_mask = (
    (df[target_col] < lower_bound) |
    (df[target_col] > upper_bound)
)

# Count outliers by station
station_outliers = (
    df.groupby("Station")
      .agg(
          total_records=(target_col, "size"),
          outlier_count=(target_col, lambda x: (
              (x < lower_bound) | (x > upper_bound)
          ).sum())
      )
      .reset_index()
)

station_outliers["outlier_percent"] = (
    station_outliers["outlier_count"] /
    station_outliers["total_records"] * 100
)

station_outliers = station_outliers.sort_values(
    "outlier_percent",
    ascending=False
)

station_outliers

,Station,total_records,outlier_count,outlier_percent
12,K Narayanapura,3317,3168,95.507989
3,Avalahalli,3110,2374,76.334405
10,Jakkur_1,3721,1199,32.222521
5,Beguru_1,3556,917,25.787402
0,Adakamaranahalli,4596,799,17.384682
13,Kethohalli,4656,633,13.595361
4,Bagalagunte,4684,580,12.382579
22,Thimmenahalli,4583,564,12.306350
1,Anekal_1,3007,205,6.817426
2,Attibele_1,3594,124,3.450195


## 8. Station-wise Target Distribution Analysis

The station-wise IQR analysis shows substantial variation in outlier percentages
across monitoring stations. Some stations have very high outlier rates, while
others have none.

This indicates that groundwater levels have different distributions across
stations, making a single global IQR threshold inappropriate for automatic
outlier removal.

The flagged observations will therefore be retained for now. The next analysis
will inspect the actual target-value ranges and distributions station-wise to
distinguish legitimate station-level variation from extreme sensor anomalies.

**Decision:** Do not remove the IQR outliers based solely on the global threshold.

In [12]:
station_target_stats = (
    df.groupby("Station")[target_col]
      .agg(
          count="count",
          mean="mean",
          median="median",
          std="std",
          min="min",
          max="max"
      )
      .reset_index()
      .sort_values("max", ascending=False)
)

station_target_stats

,Station,count,mean,median,std,min,max
0,Adakamaranahalli,4596,1789.968896,-16.0345,112053.279262,-88.658,7563528.000
22,Thimmenahalli,4583,-7.106980,-15.9370,26.144122,-38.206,79.026
13,Kethohalli,4656,-4.546084,-9.5200,22.656849,-48.608,55.003
4,Bagalagunte,4684,-3.252851,-7.2905,21.556492,-54.137,54.911
5,Beguru_1,3556,-0.575674,-18.4270,29.944540,-26.166,50.594
6,Byadarahalli,4612,-31.448407,-34.2780,9.047305,-37.956,48.363
14,Laggere,4600,-12.557863,-12.7905,6.313777,-56.810,32.750
19,Sadashivanagara,4385,-6.078667,-6.5150,2.255326,-9.223,12.457
15,Mahadevapura_2,3485,-3.245950,-4.0770,1.892390,-4.535,2.022
8,Devarabeesanahalli_1,3414,-3.660423,1.0000,5.337977,-43.081,1.000


## 9. Extreme Value & Suspicious Pattern Investigation

Station-wise statistics reveal that the global IQR outliers contain both
station-specific variation and extreme anomalous values.

A major anomaly is observed at Adakamaranahalli, where the maximum groundwater
level reaches 7,563,528 m while the median is only -16.03 m. Several stations
also contain repeated maximum values of exactly 1.000 m.

Before removing values, the extreme observations will be inspected directly
to determine whether they are isolated sensor/data-entry anomalies or recurring
patterns.

**Decision:** Investigate extreme values and suspicious repeated values before
performing any outlier treatment.

In [13]:
# Inspect the highest target values
extreme_values = (
    df[[ "Station", "Data Acquisition Time", target_col ]]
    .sort_values(target_col, ascending=False)
    .head(30)
)

extreme_values

,Station,Data Acquisition Time,Groundwater Level Telemetry 6 Hourly (meter)
19752,Adakamaranahalli,06-12-2022 12:00,7563528.000
19286,Adakamaranahalli,23-06-2022 00:00,708731.600
55457,Thimmenahalli,23-04-2023 12:00,79.026
55445,Thimmenahalli,20-04-2023 12:00,79.021
55461,Thimmenahalli,24-04-2023 12:00,79.014
55453,Thimmenahalli,22-04-2023 12:00,79.009
55449,Thimmenahalli,21-04-2023 12:00,79.009
55441,Thimmenahalli,19-04-2023 12:00,79.008
55580,Thimmenahalli,06-06-2023 12:00,79.007
55576,Thimmenahalli,04-06-2023 12:00,79.005


### Step 8: Outlier Treatment Strategy

The target-value analysis revealed substantial station-wise variation and several extreme anomalies. Therefore, global IQR-based deletion is not appropriate.

The identified extreme values will be treated conservatively:
- Preserve valid groundwater-level observations.
- Remove only clearly unrealistic sensor/data-entry anomalies.
- Avoid deleting all statistical outliers, as some may represent genuine station conditions.
- Recheck the target distribution after treatment.

In [14]:
# Step 8: Conservative treatment of clearly invalid target values

target_col = "Groundwater Level Telemetry 6 Hourly (meter)"

# Flag clearly unrealistic extreme values
invalid_mask = df[target_col].abs() > 1000

print("Clearly invalid target values:", invalid_mask.sum())

# Remove only clearly corrupted/extreme observations
df_clean = df.loc[~invalid_mask].copy()

print("Records before treatment:", len(df))
print("Records after treatment:", len(df_clean))
print("Records removed:", len(df) - len(df_clean))

# Verify target values after treatment
print("\nTarget range after treatment:")
print("Minimum:", df_clean[target_col].min())
print("Maximum:", df_clean[target_col].max())

Clearly invalid target values: 2
Records before treatment: 100881
Records after treatment: 100879
Records removed: 2

Target range after treatment:
Minimum: -190.106
Maximum: 79.026


### Step 9: Post-Treatment Target Verification

After removing only the two clearly unrealistic extreme values, the target variable was rechecked to confirm that no obvious corruption remained and that valid station-level variation was preserved.

In [15]:
# Step 9: Verify cleaned target distribution

print("Cleaned Target Statistics:")
print(df_clean[target_col].describe())

print("\nMissing target values:", df_clean[target_col].isna().sum())
print("Remaining values beyond ±1000:", (df_clean[target_col].abs() > 1000).sum())

Cleaned Target Statistics:
count    100879.000000
mean        -25.156135
std          35.985168
min        -190.106000
25%         -28.533000
50%         -18.285000
75%         -10.420000
max          79.026000
Name: Groundwater Level Telemetry 6 Hourly (meter), dtype: float64

Missing target values: 0
Remaining values beyond ±1000: 0


### Step 10: Timestamp Quality Check

The timestamp column is checked for valid datetime conversion, missing timestamps, duplicate timestamps, and chronological consistency. This ensures the dataset is suitable for time-series analysis and forecasting.

In [16]:
# Step 10: Timestamp quality check

time_col = "Data Acquisition Time"

# Convert to datetime
df_clean[time_col] = pd.to_datetime(
    df_clean[time_col],
    dayfirst=True,
    errors="coerce"
)

print("Missing/invalid timestamps:", df_clean[time_col].isna().sum())
print("Duplicate Station + Timestamp:",
      df_clean.duplicated(subset=["Station", time_col]).sum())

print("\nTimestamp range:")
print("First:", df_clean[time_col].min())
print("Last:", df_clean[time_col].max())

Missing/invalid timestamps: 0
Duplicate Station + Timestamp: 1434

Timestamp range:
First: 2021-09-27 00:00:00
Last: 2025-12-30 18:00:00


### Step 11: Duplicate Timestamp Resolution

The timestamp check found 1,434 duplicate Station + Timestamp combinations. Since each station should have one groundwater observation for a given timestamp, duplicates must be resolved before time-series analysis.

Duplicate observations will be consolidated by retaining the mean target value for each Station + Timestamp combination.

In [17]:
# Step 11: Resolve duplicate Station + Timestamp records

before = len(df_clean)

df_clean = (
    df_clean
    .groupby(["Station", time_col], as_index=False)
    .agg({
        "Latitude": "first",
        "Longitude": "first",
        "Groundwater Level Telemetry 6 Hourly (meter)": "mean",
        "RL_MSL": "first"
    })
)

after = len(df_clean)

print("Records before duplicate resolution:", before)
print("Records after duplicate resolution:", after)
print("Duplicate records consolidated:", before - after)

print(
    "Remaining Station + Timestamp duplicates:",
    df_clean.duplicated(subset=["Station", time_col]).sum()
)

Records before duplicate resolution: 100879
Records after duplicate resolution: 99445
Duplicate records consolidated: 1434
Remaining Station + Timestamp duplicates: 0


### Step 12: 6-Hourly Time-Series Consistency

The cleaned dataset is checked for consistency with the expected 6-hour observation interval at each station. Missing time intervals are identified to assess temporal gaps before time-series modeling and feature engineering.

In [18]:
# Step 12: Check 6-hourly time consistency

df_clean = df_clean.sort_values(["Station", time_col]).reset_index(drop=True)

# Calculate time difference between consecutive observations within each station
df_clean["time_diff"] = (
    df_clean.groupby("Station")[time_col]
    .diff()
)

# Expected interval
expected_interval = pd.Timedelta(hours=6)

# Count non-6-hour intervals
irregular_intervals = (df_clean["time_diff"].notna()) & (
    df_clean["time_diff"] != expected_interval
)

print("Total records:", len(df_clean))
print("Expected interval:", expected_interval)
print("Irregular intervals:", irregular_intervals.sum())

# Show the largest gaps
gap_summary = (
    df_clean.loc[irregular_intervals, ["Station", time_col, "time_diff"]]
    .sort_values("time_diff", ascending=False)
)

print("\nLargest time gaps:")
display(gap_summary.head(20))

Total records: 99445
Expected interval: 0 days 06:00:00
Irregular intervals: 4043

Largest time gaps:


,Station,Data Acquisition Time,time_diff
39745,Jakkur_1,2023-01-30 00:00:00,273 days 06:00:00
84193,Tavarekere_1,2022-05-29 00:00:00,243 days 06:00:00
64040,Marenahalli_1,2023-06-08 12:00:00,178 days 18:00:00
84315,Tavarekere_1,2022-12-07 06:00:00,156 days 00:00:00
8237,Attibele_1,2023-08-23 06:00:00,142 days 00:00:00
66723,Rajanukunte,2022-03-26 12:00:00,128 days 01:00:00
45524,Jalahalli_2,2024-08-17 00:00:00,116 days 06:00:00
85747,Tavarekere_1,2024-08-17 00:00:00,110 days 06:00:00
5252,Anekal_1,2023-09-06 00:00:00,89 days 06:00:00
45924,Jalahalli_2,2025-03-17 00:00:00,82 days 06:00:00


### Step 13: Station-wise Temporal Gap Analysis

The dataset contains 4,043 intervals that differ from the expected 6-hour frequency, including several long multi-day/month gaps. Since blindly imputing these gaps could introduce artificial patterns, the gaps are analyzed station-wise before deciding the appropriate treatment.

In [19]:
# Step 13: Station-wise temporal gap analysis

gap_analysis = (
    df_clean[df_clean["time_diff"].notna()]
    .groupby("Station")["time_diff"]
    .agg(
        total_intervals="count",
        regular_6hr=lambda x: (x == pd.Timedelta(hours=6)).sum(),
        irregular=lambda x: (x != pd.Timedelta(hours=6)).sum(),
        largest_gap="max"
    )
    .reset_index()
)

gap_analysis["irregular_percent"] = (
    gap_analysis["irregular"] /
    gap_analysis["total_intervals"] * 100
)

gap_analysis = gap_analysis.sort_values(
    "irregular_percent",
    ascending=False
)

display(gap_analysis)

,Station,total_intervals,regular_6hr,irregular,largest_gap,irregular_percent
23,Thotagere,3071,2895,176,72 days 06:00:00,5.731032
3,Avalahalli,3109,2935,174,16 days 06:00:00,5.596655
8,Devarabeesanahalli_1,3413,3240,173,72 days 06:00:00,5.068854
14,Laggere,4599,4367,232,31 days 06:00:00,5.044575
10,Jakkur_1,3720,3546,174,273 days 06:00:00,4.677419
2,Attibele_1,3118,2979,139,142 days 00:00:00,4.457986
15,Mahadevapura_2,3484,3329,155,72 days 06:00:00,4.448909
5,Beguru_1,3555,3403,152,72 days 06:00:00,4.275668
11,Jalahalli_2,3879,3718,161,116 days 06:00:00,4.150554
9,Doddakannahalli,4344,4166,178,38 days 06:00:00,4.097606


### Step 14: Final Dataset Validation

Before proceeding to feature engineering, the cleaned dataset is validated to ensure that the major cleaning operations were successful.

The final validation checks:
- Missing values in important columns
- Duplicate Station + Timestamp combinations
- Invalid timestamps
- Target-value validity
- Final record count
- Number of stations

The dataset should contain no unresolved duplicates, missing target values, or invalid timestamps.

In [20]:
# Final validation of cleaned dataset

print("=== FINAL DATA VALIDATION ===")

print(f"Total records: {len(df_clean):,}")
print(f"Number of stations: {df_clean['Station'].nunique()}")

print("\nMissing values:")
print(df_clean[
    ['Station', 'Data Acquisition Time',
     'Groundwater Level Telemetry 6 Hourly (meter)']
].isnull().sum())

print("\nDuplicate Station + Timestamp:")
print(
    df_clean.duplicated(
        subset=['Station', 'Data Acquisition Time']
    ).sum()
)

print("\nInvalid timestamps:")
print(df_clean['Data Acquisition Time'].isna().sum())

target = 'Groundwater Level Telemetry 6 Hourly (meter)'

print("\nTarget validation:")
print(f"Missing target values: {df_clean[target].isna().sum()}")
print(f"Values beyond ±1000: {((df_clean[target] < -1000) | (df_clean[target] > 1000)).sum()}")

print("\nTarget range:")
print(f"Minimum: {df_clean[target].min():.3f}")
print(f"Maximum: {df_clean[target].max():.3f}")

=== FINAL DATA VALIDATION ===
Total records: 99,445
Number of stations: 25

Missing values:
Station                                         0
Data Acquisition Time                           0
Groundwater Level Telemetry 6 Hourly (meter)    0
dtype: int64

Duplicate Station + Timestamp:
0

Invalid timestamps:
0

Target validation:
Missing target values: 0
Values beyond ±1000: 0

Target range:
Minimum: -190.106
Maximum: 79.026


In [21]:
# Final confirmation

print("=== CLEANING COMPLETE ===")
print(f"Final records: {len(df_clean):,}")
print(f"Stations: {df_clean['Station'].nunique()}")
print(f"Missing target values: {df_clean[target].isna().sum()}")
print(f"Duplicate Station + Timestamp: {df_clean.duplicated(['Station', 'Data Acquisition Time']).sum()}")
print(f"Invalid timestamps: {df_clean['Data Acquisition Time'].isna().sum()}")

=== CLEANING COMPLETE ===
Final records: 99,445
Stations: 25
Missing target values: 0
Duplicate Station + Timestamp: 0
Invalid timestamps: 0


### Final Cleaning Insight

The dataset cleaning and validation process is complete.

- Final records: 99,445
- Stations: 25
- Invalid target anomalies removed: 2
- Duplicate Station + Timestamp records consolidated: 1,434
- Missing target values: 0
- Remaining Station + Timestamp duplicates: 0
- Invalid timestamps: 0
- Irregular time intervals: 4,043
- Irregular gaps were preserved without artificial interpolation.
- Final target range: -190.106 to 79.026

The cleaned dataset is ready for feature engineering and subsequent model development.